# Step 3 — Train Gaussian Splatting

The COLMAP work is already done: steps 1 and 2 produced a folder holding
`images/` and `sparse/0/`. This notebook only trains.

**Before you run anything:** `Runtime` → `Change runtime type` → **T4 GPU**.

Upload that step-2 folder to Drive — as a `.zip` of the whole thing, a
`.tar.gz`, or the plain folder — then paste its path into **cell 3**. That path
is the only thing you have to edit; everything after it finds `images/` and
`sparse/0/` on its own and starts training.

Rough timings on a free-tier T4: setup 10 minutes, training 45-70 minutes.

Every checkpoint is copied to Drive the moment it appears, so if Colab cuts the
session off you keep whatever finished.

## 1. GPU

In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU attached. Runtime > Change runtime type > T4 GPU.")

print("GPU:", gpu.stdout.strip())

# CUDA compute capability, needed when compiling the rasteriser.
name = gpu.stdout.lower()
if "t4" in name:
    ARCH = "7.5"
elif "l4" in name:
    ARCH = "8.9"
elif "a100" in name:
    ARCH = "8.0"
else:
    ARCH = "7.5"
    print("Unknown card — assuming 7.5. If compilation fails, set ARCH by hand.")
print("ARCH =", ARCH)

## 2. Google Drive

In [ ]:
import os

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
print("Drive mounted.")

## 3. Cấu hình — chỉ sửa MỘT dòng ở đây

**Configuration — the only cell you edit.** Paste the path to your step-2 result
into `COLMAP_INPUT`, run every cell, and leave it alone. The project name is
taken from the file name, so `Mèo.zip` writes `Meo_30000.ply`.

In [ ]:
# ============ SỬA DÒNG NÀY / EDIT THIS ONE LINE ============
#
#  ┌───────────────────────────────────────────────────────────────────┐
#  │  Dán đường dẫn tới kết quả bước 2 trên Drive vào giữa hai dấu " " │
#  │  Paste the path to your step-2 result on Drive between the quotes │
#  └───────────────────────────────────────────────────────────────────┘
#
#  Lấy đường dẫn: bấm biểu tượng thư mục ở thanh bên trái Colab, mở
#  drive/MyDrive, tìm file, bấm ba chấm, "Copy path", dán vào đây.
#
#  Nhận cả ba dạng — all three shapes work:
#       Mèo.zip          zip của cả thư mục Mèo_3d          (thường dùng nhất)
#       Mèo_3d.tar.gz
#       Mèo_3d           thư mục để thẳng trên Drive
#
#  Bên trong có gì cũng được. Notebook tự tìm images/ và sparse/0/ ở bất kỳ
#  tầng nào, rồi chỉ lấy đúng hai thứ đó. Phần còn lại của bước 2 —
#  distorted/database.db (vài GB), stereo/, run-colmap-*.sh — bị bỏ qua,
#  không giải nén, vì training không đọc tới.

COLMAP_INPUT = "/content/drive/MyDrive/img3dpl/CHANGE_ME.zip"

# ===========================================================
#
# Mọi thứ dưới đây có giá trị mặc định dùng được ngay — chỉ sửa khi cần.
# Everything below has a working default. Change only if you need to.

# Tên ngắn cho lần quét này, dùng đặt tên file .ply đầu ra.
# Để trống = tự lấy theo tên file ở trên ("Mèo.zip" -> "Meo").
PROJECT = ""

# Thư mục trên Drive chứa file .ply hoàn chỉnh. Tự tạo nếu chưa có.
OUTPUT_DIR = "/content/drive/MyDrive/img3dpl/results"

# Các mốc lưu checkpoint. Bỏ bớt mốc cuối nếu Colab hay ngắt session.
SAVES = [7000, 15000, 30000]

# True thêm hai tham số làm chậm tốc độ sinh Gaussian mới.
# Cần cho cảnh nhiều chi tiết (trên 100.000 điểm ban đầu) vì T4 chỉ có 15 GB VRAM.
LIMIT_VRAM = True

# Bậc spherical harmonics cho bản nén: 1 nhẹ, 3 giữ đủ phản chiếu nhưng nặng.
COMPRESS_SH_DEGREE = 1

# ========================================================
import os
import re
import unicodedata

if "CHANGE_ME" in COLMAP_INPUT:
    raise SystemExit("Sửa COLMAP_INPUT ở trên thành đường dẫn của chính bạn.")

if not os.path.exists(COLMAP_INPUT):
    raise SystemExit(f"Không thấy trên Drive / not found: {COLMAP_INPUT}")


def short_name(path):
    """Tên dự án lấy từ tên file: "Mèo.zip" -> "Meo".

    Bỏ dấu tiếng Việt và mọi ký tự lạ, vì tên này đi vào tên file .ply và
    vào dòng lệnh — tên có dấu hay khoảng trắng là nguồn lỗi khó đoán.
    """
    name = os.path.basename(path.rstrip("/"))
    for suffix in (".tar.gz", ".tgz", ".tar", ".zip"):
        if name.lower().endswith(suffix):
            name = name[: -len(suffix)]
            break
    name = name.replace("đ", "d").replace("Đ", "D")
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
    return name or "scan"


PROJECT = PROJECT.strip() or short_name(COLMAP_INPUT)
ITERATIONS = max(SAVES)
WORK = f"/content/{PROJECT}"

print("Đầu vào / input :", COLMAP_INPUT)
print("Tên dự án       :", PROJECT)
print("Đầu ra / output :", OUTPUT_DIR)
print("Checkpoints     :", SAVES)

## 4. Find the scene and load it

Runs on its own — nothing to edit here.

Step 2 leaves a folder like this, and you can hand the whole thing over as a
`.zip`, a `.tar.gz`, or a plain folder:

```
Mèo_3d/
├── images/                 ← used: undistorted photos
├── sparse/0/*.bin          ← used: cameras, poses, sparse cloud
├── distorted/database.db   ← skipped, and it is the biggest file by far
├── distorted/sparse/0/     ← skipped: the model before undistortion
├── stereo/                 ← skipped: empty scaffolding for dense
└── run-colmap-*.sh         ← skipped
```

Only `images/` and `sparse/0/` are read out of the archive; everything else is
left inside it. That saves several gigabytes of Colab disk and a lot of waiting,
because `database.db` alone is usually larger than the rest put together.

The scene is located by looking for a folder that holds **both** `images/` and
`sparse/0/`, at any depth — so an extra wrapper folder inside the zip is fine,
and `distorted/sparse/0/` is never mistaken for the real one.

In [ ]:
import os
import shutil
import struct
import tarfile
import time
import zipfile

KEEP = ("images", "sparse")     # the only two folders train.py ever reads

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)
started = time.time()


def find_scene(paths):
    """The folder holding both images/ and sparse/0/. "" means the top level.

    Step 2 writes a second model under distorted/sparse/0 — the one from before
    undistortion, whose poses do not line up with the photos in images/. Asking
    for images/ as a sibling is what rules it out: only the real scene has one.
    """
    with_sparse, with_images = set(), set()
    for path in paths:
        parts = path.strip("/").split("/")
        for i, part in enumerate(parts[:-1]):
            prefix = "/".join(parts[:i])
            if part == "sparse" and parts[i + 1] == "0":
                with_sparse.add(prefix)
            elif part == "images":
                with_images.add(prefix)
    both = with_sparse & with_images
    return min(both, key=len) if both else None


def inside_scene(path, scene):
    """Path rewritten relative to the scene, or None if training does not want it."""
    parts = path.strip("/").split("/")
    prefix = scene.split("/") if scene else []
    if parts[:len(prefix)] != prefix:
        return None
    rest = parts[len(prefix):]
    if len(rest) < 2 or rest[0] not in KEEP:
        return None
    return "/".join(rest)


def give_up(paths):
    """No scene found — show what the input actually holds, then stop."""
    print("\nKhông tìm thấy thư mục nào chứa cả images/ và sparse/0/.")
    print("No folder inside holds both images/ and sparse/0/. What is in there:\n")
    folders = {}
    for path in paths:
        folders.setdefault(os.path.dirname(path), []).append(os.path.basename(path))
    for folder in sorted(folders)[:25]:
        names = sorted(folders[folder])
        print(f"  {folder or '.'}/  ({len(names)} files)  {', '.join(names[:4])}")
    raise SystemExit("Sai file? Cần kết quả của bước 2, không phải thư mục ảnh gốc.")


def write(fileobj, relative):
    """Save one file from the archive into WORK, keeping images/ and sparse/."""
    target = os.path.join(WORK, relative)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    with open(target, "wb") as out:
        shutil.copyfileobj(fileobj, out)


def size_text(count):
    """Bytes as GB once GB is worth saying, otherwise MB."""
    return f"{count / 1e9:.2f} GB" if count >= 1e9 else f"{count / 1e6:.0f} MB"


taken = skipped = 0          # file counts
taken_size = skipped_size = 0

if os.path.isdir(COLMAP_INPUT):
    print("Đầu vào là thư mục — đang chép sang đĩa của Colab...")
    # Copy rather than train straight off Drive: training reads these files
    # thousands of times, and Drive is far slower than local disk.
    listing = []
    for folder, _, files in os.walk(COLMAP_INPUT):
        rel = os.path.relpath(folder, COLMAP_INPUT)
        rel = "" if rel == "." else rel
        listing += [os.path.join(rel, name) for name in files]

    scene = find_scene(listing)
    if scene is None:
        give_up(listing)

    for path in listing:
        source = os.path.join(COLMAP_INPUT, path)
        size = os.path.getsize(source)
        relative = inside_scene(path, scene)
        if relative is None:
            skipped, skipped_size = skipped + 1, skipped_size + size
            continue
        with open(source, "rb") as handle:
            write(handle, relative)
        taken, taken_size = taken + 1, taken_size + size

elif COLMAP_INPUT.lower().endswith(".zip"):
    print("Đầu vào là .zip — đang giải nén phần cần thiết...")
    with zipfile.ZipFile(COLMAP_INPUT) as archive:
        entries = [e for e in archive.infolist() if not e.is_dir()]
        scene = find_scene([e.filename for e in entries])
        if scene is None:
            give_up([e.filename for e in entries])

        for entry in entries:
            relative = inside_scene(entry.filename, scene)
            if relative is None:
                skipped, skipped_size = skipped + 1, skipped_size + entry.file_size
                continue
            with archive.open(entry) as handle:
                write(handle, relative)
            taken, taken_size = taken + 1, taken_size + entry.file_size

else:
    print("Đầu vào là .tar.gz — đang giải nén phần cần thiết...")
    with tarfile.open(COLMAP_INPUT) as archive:
        members = [m for m in archive.getmembers() if m.isfile()]
        scene = find_scene([m.name for m in members])
        if scene is None:
            give_up([m.name for m in members])

        for member in members:
            relative = inside_scene(member.name, scene)
            if relative is None:
                skipped, skipped_size = skipped + 1, skipped_size + member.size
                continue
            write(archive.extractfile(member), relative)
            taken, taken_size = taken + 1, taken_size + member.size

print(f"Thư mục cảnh / scene folder: {scene or '(gốc / top level)'}")
print(f"Đã lấy  : {taken} file, {size_text(taken_size)}")
print(f"Bỏ qua  : {skipped} file, {size_text(skipped_size)} "
      f"(distorted/, stereo/, script — training không đọc tới)")
print(f"Xong sau {time.time() - started:.0f}s")

# ---- Check that what came out is actually trainable -------------------------
sparse = f"{WORK}/sparse/0"
photos = sorted(f for f in os.listdir(f"{WORK}/images")
                if f.lower().endswith((".jpg", ".jpeg", ".png")))

if not photos:
    raise SystemExit("images/ rỗng — kiểm tra lại file đã tải lên Drive.")

for stem in ("cameras", "images", "points3D"):
    if not (os.path.exists(f"{sparse}/{stem}.bin")
            or os.path.exists(f"{sparse}/{stem}.txt")):
        raise SystemExit(f"Thiếu sparse/0/{stem}.bin — bước 2 chưa chạy xong?")

print(f"\nẢnh / images : {len(photos)}")
print("sparse/0     :", ", ".join(sorted(os.listdir(sparse))))

# images.bin starts with a uint64: how many photos COLMAP actually placed.
# A model built from 320 photos that only registered 90 of them trains fine and
# then looks wrong, so it is worth saying out loud before an hour of training.
if os.path.exists(f"{sparse}/images.bin"):
    with open(f"{sparse}/images.bin", "rb") as handle:
        registered = struct.unpack("<Q", handle.read(8))[0]
    print(f"Đã định vị   : {registered}/{len(photos)} ảnh")
    if registered < 0.7 * len(photos):
        print("!! Nhiều ảnh không được định vị. Model sẽ thiếu mảng.")
        print("!! Ảnh cạnh nhau cần trùng nhau khoảng 60-80%.")

## 5. Install Gaussian Splatting

Takes about 10 minutes, most of it compiling the CUDA rasteriser. Output is kept
quiet unless something fails, in which case the last 30 lines are printed.

In [ ]:
import os
import subprocess

os.environ["TORCH_CUDA_ARCH_LIST"] = ARCH
REPO = "/content/gaussian-splatting"


def step(label, command):
    """Run a shell command quietly. On failure, show enough to diagnose it."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED: {label}\n")
        print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))
        raise RuntimeError(label)
    print("ok  ", label)


if not os.path.isdir(REPO):
    step("clone repository",
         "git clone -q --recursive "
         f"https://github.com/graphdeco-inria/gaussian-splatting {REPO}")

step("plyfile", "pip -q install plyfile")
step("diff-gaussian-rasterization",
     f"pip -q install {REPO}/submodules/diff-gaussian-rasterization")
step("simple-knn", f"pip -q install {REPO}/submodules/simple-knn")

if os.path.isdir(f"{REPO}/submodules/fused-ssim"):
    step("fused-ssim", f"pip -q install {REPO}/submodules/fused-ssim")

import torch
from diff_gaussian_rasterization import GaussianRasterizer  # noqa: F401

print()
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Ready to train.")

## 6. Train

Each checkpoint is copied to Drive as soon as it lands on disk. If Colab drops
the session halfway, everything already copied is still yours.

A progress line prints every 30 seconds so you can tell it is alive.

In [ ]:
import glob
import os
import shutil
import subprocess
import time

os.makedirs(OUTPUT_DIR, exist_ok=True)

command = ["python", "train.py",
           "-s", WORK,
           "-m", f"{WORK}/output",
           "--iterations", str(ITERATIONS),
           "--save_iterations"] + [str(s) for s in SAVES]

if LIMIT_VRAM:
    command += ["--densify_grad_threshold", "0.0004",
                "--densify_until_iter", "12000"]

log_path = f"{WORK}/train.log"
log_file = open(log_path, "w")
trainer = subprocess.Popen(command, cwd="/content/gaussian-splatting",
                           stdout=log_file, stderr=subprocess.STDOUT, text=True)
print("Training started. A progress line follows every 30 seconds.\n")

copied = set()


def copy_new_checkpoints():
    """Copy any checkpoint that has appeared since the last look."""
    pattern = f"{WORK}/output/point_cloud/iteration_*/point_cloud.ply"
    for path in glob.glob(pattern):
        step_number = path.split("iteration_")[1].split("/")[0]
        if step_number in copied:
            continue
        time.sleep(5)          # let the file finish being written
        target = f"{OUTPUT_DIR}/{PROJECT}_{step_number}.ply"
        shutil.copy(path, target)
        copied.add(step_number)
        size = os.path.getsize(target) / 1e6
        print(f">>> SAVED TO DRIVE: {PROJECT}_{step_number}.ply ({size:.0f} MB)")


while trainer.poll() is None:
    time.sleep(30)
    copy_new_checkpoints()
    tail = subprocess.run(["tail", "-1", log_path],
                          capture_output=True, text=True).stdout.strip()
    if tail:
        print(tail)

log_file.close()
copy_new_checkpoints()

if not copied:
    print("\nNo checkpoint was produced. Last 30 lines of the log:\n")
    print(subprocess.run(["tail", "-30", log_path],
                         capture_output=True, text=True).stdout)
else:
    print("\nFinished. Saved:", sorted(copied, key=int))

## 7. Make a lighter copy

Throws away the near-transparent blobs and lowers the spherical-harmonics degree
from 3 to `COMPRESS_SH_DEGREE`. The lighter file sits beside the original on
Drive, ending in `_light.ply`.

Useful because the full file often will not open on a modest laptop.

In [ ]:
import glob
import os

import numpy as np
from plyfile import PlyData, PlyElement


def compress_ply(source, target, sh_degree=1, opacity_threshold=0.05):
    """Drop faint Gaussians and trim the colour detail. Returns (kept, total)."""
    vertices = PlyData.read(source)["vertex"]

    # Stored opacity is pre-sigmoid, so convert before comparing.
    opacity = 1 / (1 + np.exp(-np.asarray(vertices["opacity"])))
    keep = opacity > opacity_threshold

    rest_per_channel = {0: 0, 1: 3, 2: 8, 3: 15}[sh_degree]

    src_names = ["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1", "f_dc_2"]
    dst_names = list(src_names)

    index = 0
    for channel in range(3):
        for i in range(rest_per_channel):
            src_names.append(f"f_rest_{channel * 15 + i}")
            dst_names.append(f"f_rest_{index}")
            index += 1

    tail = ["opacity", "scale_0", "scale_1", "scale_2",
            "rot_0", "rot_1", "rot_2", "rot_3"]
    src_names += tail
    dst_names += tail

    out = np.empty(int(keep.sum()), dtype=[(n, "f4") for n in dst_names])
    for src, dst in zip(src_names, dst_names):
        out[dst] = np.asarray(vertices[src])[keep]

    PlyData([PlyElement.describe(out, "vertex")]).write(target)
    return int(keep.sum()), len(keep)


for source in sorted(glob.glob(f"{OUTPUT_DIR}/{PROJECT}_*.ply")):
    if "_light" in source:
        continue
    target = source.replace(".ply", "_light.ply")
    kept, total = compress_ply(target=target, source=source,
                               sh_degree=COMPRESS_SH_DEGREE)
    before = os.path.getsize(source) / 1e6
    after = os.path.getsize(target) / 1e6
    print(f"{os.path.basename(target)}: kept {kept}/{total} blobs | "
          f"{before:.0f} -> {after:.0f} MB ({before / after:.1f}x smaller)")

## 8. View the result

Download a `_light.ply` from Drive, open **https://superspl.at/editor**, and drag
the file in.

The model has no real-world scale — use the editor's scale tool to size it by eye.

---

### If something went wrong

**"Không tìm thấy thư mục nào chứa cả images/ và sparse/0/"** — the file you
pointed at is not a step-2 result. The photo folder from step 1 is not enough:
training needs `sparse/0/*.bin`, which only step 2 produces. Cell 4 prints what
your file actually contained, which usually makes it obvious.

**Uploading took forever** — you probably zipped `distorted/database.db` along
with everything else. It is several gigabytes and training never opens it. Leave
it out next time:
`zip -r -0 Meo.zip Meo_3d -x "Meo_3d/distorted/*" "Meo_3d/stereo/*"`

**Out of VRAM during training** — set `LIMIT_VRAM = True` if it is not already,
or drop the last entry from `SAVES`.

**The light copy looks flat, reflections gone** — set `COMPRESS_SH_DEGREE = 3`
and run cell 7 again. The full file is still on Drive; no need to retrain.

**Colab disconnected halfway** — run again from cell 1. Checkpoints already
copied to Drive are still good.

**Only a few images were reconstructed in step 2** — cell 4 tells you this
before training starts (`Đã định vị : 90/320 ảnh`). The problem is the photos,
not the training. Neighbouring shots need roughly 60-80% overlap.